# 06 — Perspective Transform (Pixel → Real-World Metres)

A camera looking at a football pitch at an angle introduces **perspective distortion**: players near the top of the frame appear smaller and closer together than players near the bottom, even if they're the same distance apart on the pitch.

A **homography** (a 3×3 matrix) corrects this: it maps any 4 pixel points to 4 real-world coordinates, and lets us convert *any* pixel in between.

**Connection to the pipeline:** `football_analysis/view_transformer/view_transformer.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
VIDEO_PATH = '../input_videos/08fd33_4.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame0 = cap.read()
cap.release()

## The calibration points

We manually identify 4 points in the video frame that correspond to known real-world positions on the pitch.  
Here we used the four corners of a visible pitch section — measured from official pitch markings.

These are specific to this camera angle and clip (`08fd33_4.mp4`).

In [ ]:
# Pixel coordinates (manually calibrated for this camera angle)
pixel_vertices = np.array([
    [110,  1035],   # bottom-left
    [265,   275],   # top-left
    [910,   260],   # top-right
    [1640,  915],   # bottom-right
], dtype=np.float32)

# Corresponding real-world coordinates in metres
# This section of pitch is 23.32 m deep × 68 m wide
PITCH_LENGTH = 23.32   # metres (depth visible in this clip)
PITCH_WIDTH  = 68.0    # metres (full pitch width)

world_vertices = np.array([
    [0,            PITCH_WIDTH],   # bottom-left
    [0,            0           ],   # top-left
    [PITCH_LENGTH, 0           ],   # top-right
    [PITCH_LENGTH, PITCH_WIDTH ],   # bottom-right
], dtype=np.float32)

print('Pixel vertices (x, y):')
print(pixel_vertices)
print()
print('World vertices (metres):')
print(world_vertices)

In [ ]:
# Show the calibration quad on the frame
vis = frame0.copy()
pts = pixel_vertices.astype(int)
cv2.polylines(vis, [pts], isClosed=True, color=(0, 255, 0), thickness=3)
labels = ['BL', 'TL', 'TR', 'BR']
for pt, label in zip(pts, labels):
    cv2.circle(vis, tuple(pt), 8, (0, 0, 255), -1)
    cv2.putText(vis, label, tuple(pt + [-30, -10]), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

plt.figure(figsize=(16, 9))
plt.imshow(vis[:, :, ::-1])
plt.title('Calibration quadrilateral (green) — these 4 pixels map to known pitch coordinates')
plt.axis('off')
plt.show()

## Computing the homography matrix

`cv2.getPerspectiveTransform` computes the 3×3 matrix **H** such that for any pixel point **p**, the real-world point **q = H × p** (in homogeneous coordinates).

In [ ]:
H = cv2.getPerspectiveTransform(pixel_vertices, world_vertices)
print('Homography matrix H:')
print(H)

## Transforming a point

`cv2.perspectiveTransform` applies **H** to a set of points.  
We first verify the calibration points themselves round-trip correctly.

In [ ]:
# Verify calibration corners
test_pts = pixel_vertices.reshape(-1, 1, 2)
transformed = cv2.perspectiveTransform(test_pts, H).reshape(-1, 2)

print('Calibration check (should match world_vertices):')
for pixel, world, result in zip(pixel_vertices, world_vertices, transformed):
    print(f'  pixel {pixel} → expected {world} → got {result.round(2)}')

## Transforming player foot positions

The foot position (bottom-centre of bbox) is the player's contact point with the pitch.  
We check with `pointPolygonTest` that the point lies inside the calibration quad before transforming — points outside the quad produce unreliable results.

In [ ]:
def transform_point(pixel_point, H, polygon):
    """
    Transform a pixel (x, y) to real-world (x, y) in metres.
    Returns None if the point is outside the calibration polygon.
    """
    inside = cv2.pointPolygonTest(polygon, (float(pixel_point[0]), float(pixel_point[1])), False)
    if inside < 0:
        return None
    pt = np.array([[pixel_point]], dtype=np.float32)
    result = cv2.perspectiveTransform(pt, H)
    return result[0][0].tolist()

polygon = pixel_vertices.reshape((-1, 1, 2)).astype(np.float32)

# Test a point roughly in the centre of the pitch section visible
test_pixel = (875, 647)   # approx centre of the calibration quad
world_pos  = transform_point(test_pixel, H, polygon)
print(f'Pixel {test_pixel} → World {[round(v, 2) for v in world_pos]} m')

# Test a point outside the quad
outside_pixel = (10, 10)
print(f'Pixel {outside_pixel} → {transform_point(outside_pixel, H, polygon)}')

## Visualising the bird's-eye view

`cv2.warpPerspective` applies the homography to the entire image, producing a flat top-down view of the pitch section.

In [ ]:
# Scale metres → pixels for display (20 px per metre)
SCALE = 20
out_w = int(PITCH_LENGTH * SCALE)
out_h = int(PITCH_WIDTH  * SCALE)

H_scaled = cv2.getPerspectiveTransform(
    pixel_vertices,
    world_vertices * SCALE
)

warped = cv2.warpPerspective(frame0, H_scaled, (out_w, out_h))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(frame0[:, :, ::-1])
axes[0].set_title('Original camera view')
axes[0].axis('off')

axes[1].imshow(warped[:, :, ::-1])
axes[1].set_title('Bird\'s-eye (perspective-corrected) view')
axes[1].set_xlabel(f'{PITCH_LENGTH} m')
axes[1].set_ylabel(f'{PITCH_WIDTH} m')

plt.suptitle('Homography: camera view → real-world top-down', fontsize=13)
plt.tight_layout()
plt.show()

## Why only some players get speed labels

Players near the frame edges or behind the goal stand outside the calibration quad.  
`pointPolygonTest` returns `None` for them, so speed is not computed.  
This is intentional — extrapolating beyond the calibrated region would give inaccurate distances.

In [ ]:
from football_analysis.trackers.tracker import Tracker
from football_analysis.view_transformer.view_transformer import ViewTransformer

tracker = Tracker('../models/best.pt')
tracks  = tracker.get_object_tracks([], read_from_stub=True, stub_path='../stubs/track_stubs.pkl')
tracker.add_position_to_tracks(tracks)
ViewTransformer().add_transformed_position_to_tracks(tracks)

# Count how many player appearances have a valid transformed position
total = 0
has_transform = 0
for frame_tracks in tracks['players']:
    for info in frame_tracks.values():
        total += 1
        if info.get('position_transformed') is not None:
            has_transform += 1

print(f'Player appearances with valid position: {has_transform}/{total} ({has_transform/total*100:.1f}%)')
print(f'Outside calibration quad (no speed):    {total - has_transform}')

## Key takeaways

| Concept | Detail |
|---|---|
| Homography | 3×3 matrix mapping 4 pixel points to 4 real-world points |
| Perspective distortion | Players at the top of frame appear closer together (smaller pixel distance = same real distance) |
| `pointPolygonTest` | Prevents extrapolation outside the calibrated region |
| Calibration effort | Manual — these 4 points are specific to this camera angle and clip |

**Next:** `07_speed_distance.ipynb` — computing speed in km/h from real-world positions